# Lou et al. (arXiv 2610.20067) Table 1 재현 — NQ-Swap · Llama-3.1-8B-Instruct · layer 10

목표 수치(논문 Table 1, NQ Llama): Unsteered **33.7 / 49.4**, Ours **Acc_par 56.8 / Acc_ctx 55.5**.

- `SOTA/lou/upstream`(원 저자 코드)은 수정하지 않고 `SOTA/lou/scripts/`에서 호출만 합니다. 경로·인자 확인 내용은 [SOTA/lou/README.md](README.md)에 있습니다.
- 준비물: Colab Secrets의 `HF_TOKEN`(Llama 3.1 라이선스 승인 계정, 노트북 접근 허용), Drive 여유 공간 약 20 GB(모델 16 GB + dump), GPU L4 또는 A100.
- 단계별 셀은 재실행 시 완료된 산출물을 건너뜁니다. 런타임이 끊기면 설정 셀부터 다시 실행하고 이어서 진행하세요.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 설정 · 환경 변수

모든 산출물은 Drive의 `sota_lou/` 아래에 저장됩니다. 저장소는 속도를 위해 로컬 `/content`에 clone합니다.

`BATCH_SIZE`: 논문 값은 64(A100 80GB)입니다. 메모리에 따라 자동으로 줄이며, batch 크기는 left padding과 bf16 수치 때문에 greedy 출력을 미세하게 바꿀 수 있습니다(README 참고).

In [ ]:
import os, torch
from google.colab import userdata

REPO_URL = "https://github.com/Ontology0/Graduation-Project.git"
BRANCH = "sota/lou"
REPO_DIR = "/content/Graduation-Project"
LOU_DIR = f"{REPO_DIR}/SOTA/lou"
SOTA_LOU = "/content/drive/MyDrive/sota_lou"

os.environ.update({
    "HF_ENDPOINT": "https://huggingface.co",
    "MODEL_ROOT": f"{SOTA_LOU}/models",
    "DATA_ROOT": f"{SOTA_LOU}/data",
    "DUMP_ROOT": f"{SOTA_LOU}/dump",
    "STEER_OUT_ROOT": f"{SOTA_LOU}/steer_result",
    "MODEL_KEY": "llama3-8b-it",   # dump·steer가 같은 키를 써야 함 (README 참고)
    "LAYER": "10",
    "GPU": "0",
    "ALPHA_MAX": "3",
    "ALPHA_STEP": "0.2",
    "SPLIT_SEED": "42",
})
for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(k, None)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # 값은 출력하지 않음

mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH_SIZE = 16 if mem_gb >= 70 else 8 if mem_gb >= 35 else 4
os.environ["BATCH_SIZE"] = str(BATCH_SIZE)
for k in ("MODEL_ROOT", "DATA_ROOT", "DUMP_ROOT", "STEER_OUT_ROOT"):
    os.makedirs(os.environ[k], exist_ok=True)
os.makedirs(f"{SOTA_LOU}/logs", exist_ok=True)
print(f"GPU {torch.cuda.get_device_name(0)} {mem_gb:.0f} GB -> BATCH_SIZE={BATCH_SIZE}")
print({k: os.environ[k] for k in ("MODEL_ROOT", "DATA_ROOT", "DUMP_ROOT", "STEER_OUT_ROOT")})

## 4. 저장소 clone (`sota/lou`, submodule 포함)

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--recurse-submodules", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "submodule", "update", "--init", "--recursive"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print("upstream:", subprocess.run(["git", "-C", f"{LOU_DIR}/upstream", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

## 5. 패키지 설치

upstream README는 `torch transformers tokenizers numpy matplotlib`만 요구하고 버전을 고정하지 않습니다. 여기서는 Llama 3.1을 지원하고 upstream 코드(`torch_dtype=`, tuple 출력 hook)와 맞는 4.x 버전을 고정합니다. `device_map`에는 `accelerate`가 필요합니다.

In [ ]:
TRANSFORMERS_SPEC = "transformers==4.46.3"
!pip install -q "{TRANSFORMERS_SPEC}" accelerate datasets huggingface_hub numpy matplotlib
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "tokenizers", "accelerate", "datasets", "huggingface_hub")})

### 실행 helper (출력 스트리밍 + Drive 로그)

In [ ]:
import subprocess, sys, time
def run(cmd, **env):
    log = f"{SOTA_LOU}/logs/{time.strftime('%Y%m%d-%H%M%S')}_{os.path.basename(cmd[1]).split('.')[0]}.log"
    e = {**os.environ, **{k: str(v) for k, v in env.items()}}
    with open(log, "w") as f, subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            sys.stdout.write(line); f.write(line)
    print(f"[exit {p.returncode}] log: {log}")
    if p.returncode:
        raise RuntimeError(f"failed: {' '.join(cmd)}")

## 6. 모델 다운로드 → `$MODEL_ROOT/Meta-Llama-3.1-8B-Instruct`

In [ ]:
run(["bash", f"{LOU_DIR}/scripts/download_model.sh"])

## 7. NQ-Swap 준비 → `$DATA_ROOT/NQ-Swap/dev.jsonl`

In [ ]:
run(["bash", f"{LOU_DIR}/scripts/prepare_nqswap.sh"])

## 8. 실행

`run_nqswap_llama.sh`를 단계별로 호출합니다. 각 단계는 완료 산출물이 있으면 건너뜁니다. 다시 계산하려면 `FORCE=1`을 넘기세요.

### 8-1. dump (stage1 closed-book · stage2 with context · 전 layer last-token resid)

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="dump", BATCH_SIZE="16")

### 8-2. 방향 추정(train 70%) + α⁺ 탐색(val 15%, 0..3) + test 평가 → Acc_par

In [ ]:
from huggingface_hub import hf_hub_download
import os, re
dst = "/content/models/Meta-Llama-3.1-8B-Instruct"
rev = re.findall(r"[0-9a-f]{40}", open(f"{dst}/.hf_revision").read())
rev = rev[0] if rev else None
print("revision:", rev)
hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "model-00003-of-00004.safetensors",
                local_dir=dst, revision=rev, token=os.environ["HF_TOKEN"])
!ls -la {dst}/*.safetensors

In [ ]:
import os, shutil
src = "/content/drive/MyDrive/sota_lou/models/Meta-Llama-3.1-8B-Instruct"
dst = "/content/models/Meta-Llama-3.1-8B-Instruct"
if not os.path.exists(f"{dst}/model-00004-of-00004.safetensors"):
    shutil.copytree(src, dst, dirs_exist_ok=True)
!ls -la {dst}
os.environ["MODEL_ROOT"] = "/content/models"
os.environ["BATCH_SIZE"] = "16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="par")

### 8-3. α⁻ 탐색(val, −3..−0.2, Acc_ctx 기준) + test 평가 → Acc_ctx

In [ ]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="ctx")

### 8-4. Unsteered test (α=0)

In [ ]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="base")

## 9. Table 1 비교

In [ ]:
run(["bash", f"{LOU_DIR}/scripts/run_nqswap_llama.sh"], STEPS="summary")
import json
s = json.load(open(f"{os.environ['STEER_OUT_ROOT']}/table1_summary.json"))
print(json.dumps({k: s[k] for k in ("alpha_plus", "alpha_plus_if_selected_by_val_acc_par", "alpha_minus")}, indent=2))

In [ ]:
# ## 10. 추가 실험: α⁺=1.6 (val Acc_par/N 기준이었다면 선택됐을 값)
import os, glob, json, sys

A = "1.6"
D = f"{os.environ['DUMP_ROOT']}/{os.environ['MODEL_KEY']}/NQ-Swap/dev"
OUT = f"{os.environ['STEER_OUT_ROOT']}/par_a{A}"
SRC = f"{LOU_DIR}/upstream/src/run_resid_projection_ablation.py"

if not glob.glob(f"{OUT}/**/L10_alpha{A}_steer_result.json", recursive=True):
    run([sys.executable, SRC,
         "--eval-mode", "train_val_test", "--split-protocol", "same_dataset", "--split-seed", "42",
         "--train-ratio", "0.7", "--val-ratio", "0.15", "--model", os.environ["MODEL_KEY"],
         "--npz", f"{D}/resid_hidden_state__stage1_correct_use_wrong_context.npz",
         "--test-json", f"{D}/internal_aware_data__stage1_correct_use_wrong_context.json",
         "--out-root", OUT, "--output-category", "dev",
         "--direction-category-tag", "dev", "--test-category-tag", "dev",
         "--gpu", "0", "--layers", "10",
         "--alpha-start", A, "--alpha-end", A, "--alpha-step", "0.2",
         "--batch-size", os.environ["BATCH_SIZE"],
         "--max-samples-stage1", "0", "--max-samples-stage2", "0",
         "--stage1-min-pass-rate", "-1"])

# 논문 방식 재채점 (base·ctx는 기존 결과, par만 α=1.6으로 교체)
R = os.environ["STEER_OUT_ROOT"]
f = lambda p: sorted(glob.glob(p, recursive=True))[-1]
par = f(f"{OUT}/**/L10_alpha{A}_steer_result.json")
base = f(f"{R}/base/**/L10_alpha0*_steer_result.json")
ctx = f(f"{R}/ctx/**/L10_alpha-0.8_steer_result.json")
print("par :", par, "\nbase:", base, "\nctx :", ctx)
# 기존 요약(table1_summary.json)을 덮어쓰지 않도록 par_a1.6 폴더를 steer-root로 쓴다.
# 복사본의 최상위 필드(alpha_plus, steer_par 등)는 α⁺=2.4 실행 값 그대로이며, α=1.6 수치는 paper_metric.runs에 기록된다.
import shutil
if os.path.exists(f"{R}/table1_summary.json") and not os.path.exists(f"{OUT}/table1_summary.json"):
    shutil.copy(f"{R}/table1_summary.json", f"{OUT}/table1_summary.json")
run([sys.executable, f"{LOU_DIR}/scripts/rescore_paper_metric.py",
     "--steer-root", OUT,
     "--manifest", f"{D}/internal_aware_data__stage1_correct_use_wrong_context.json",
     "--base", base, "--par", par, "--ctx", ctx])